# Milestone 13 - Evaluating the tutor

The tutor in "Ask about this lesson" now has **prepared answers**: 1,728 answers to the standard
questions, written ahead so Ask works even with no model running. This notebook opens up how they were
made and checked, using the same ideas the Evals module teaches, applied to this app itself.

| Step | What happened | Where it lives |
|---|---|---|
| **Draft** | A strong model (Opus 5.5 on Databricks) wrote an answer to every standard question | `src/drill/content/prepared/` |
| **Judge** | A second model (Sonnet 5.5) checked each one: correct, answers it, plain words, no exercise answer, makes sense to anyone | the `judge` field of each answer |
| **Split** | Whole lessons were dealt into dev, test and holdout | `src/drill/content/prepared_splits.json` |
| **Re-judge** | A cheaper model (Haiku 5.5) judged a stratified sample of dev, to see if it could watch live answers | MLflow run, and a recording for this notebook |
| **Decide** | Compare the two judges with the right statistics, and work out what to do next | this notebook |

You'll write five small functions the real evaluation uses, each with tests, and read the results of the
real run. **Nothing here costs money**: the judge run is replayed from a recording. Section 9 has an
optional `LIVE` switch to look at your own MLflow.

In [ ]:
import json
import math
from collections import Counter
from pathlib import Path

import ipytest
import pandas as pd
import pytest

from drill import tutor_prepared as tp
from drill import tutor_prepared_eval as tpe
from drill.content.build import build_data

ipytest.autoconfig(clean="test_*")

DATA = build_data()                      # every lesson, as the page sees it
ANSWERS = tp.load()                      # the prepared answers: {lesson_id: {question key: record}}
CHIPS = tp.chips(DATA)                   # the standard questions, as the page asks them
REC = json.loads((Path(tp.__file__).parent / "content" / "prepared_eval" / "dev_sample_haiku.json").read_text())
PHASE = tpe.phases(DATA)                 # each lesson's phase (or the platform track)

print(f"{sum(len(v) for v in CHIPS.values())} questions across {len(CHIPS)} lessons")
print(f"{sum(len(v) for v in ANSWERS.values())} prepared answers; status: {Counter(r['status'] for v in ANSWERS.values() for r in v.values())}")
print(f"recorded judge run: {REC['params']['judge_model']} on {len(REC['answers'])} answers (dataset digest {REC['params']['dataset_digest']})")

## 1. One prepared answer, start to finish

Every answer is stored with what it was written from and what the judge said. The **fingerprint** is a
short code made from the lesson's text and the question: if the lesson changes, the fingerprint no longer
matches, and the page stops showing the answer until it is drafted again.

In [ ]:
lesson, key = "use_case_score", "learn:term"
chip = next(c for c in CHIPS[lesson] if c["key"] == key)
rec = ANSWERS[lesson][key]

print("Question:", chip["q"])
print("How the tutor was told to answer:", chip["how"][:120], "...")
print()
print("Answer (first 400 characters):")
print(rec["a"][:400], "...")
print()
print("Drafted by:", rec["model"], "| status:", rec["status"])
print("Judge said:", {k: rec["judge"][k] for k in ("correct", "answers_it", "plain", "no_giveaway", "standalone")})
print("Fingerprint still matches the lesson:", rec["fingerprint"] == tp.fingerprint(DATA, lesson, chip))
print("Shown to learners:", key in tp.served(DATA, ANSWERS).get(lesson, {}))

## 2. Splitting: whole lessons, dealt within each phase

To trust a score, the answers it is measured on must not be the ones you tuned on. That's what **dev**
(tune here), **test** (check before a release) and **holdout** (big decisions only) are for.

Two rules from the Evals module:

- **Keep things that belong together in one split** (the leakage rule). All of a lesson's questions share
  its text, so we split whole **lessons**, not single answers.
- **Stratify**: deal each group across the splits, so every split gets every group.

The first attempt used a hash of each lesson's id alone (lesson "Splits that don't move"). Here is what
that gave per phase:

In [ ]:
plain = {lid: ("dev" if tpe._bucket(lid) < 60 else "test" if tpe._bucket(lid) < 80 else "holdout") for lid in CHIPS}
table = pd.crosstab(pd.Series([PHASE.get(l, "") for l in plain], name="phase"), pd.Series(list(plain.values()), name="split"))
table[["dev", "test", "holdout"]]

**Design and Ship got no holdout lessons at all.** A plain random split can miss a small group entirely,
which is the exact problem the lesson "Dev, test and holdout sets" warns about.

**Exercise.** Deal each phase's lessons like cards, in the course's pattern: dev, dev, dev, test, holdout,
then start again. Keep a count **per phase**, so each phase starts its own pattern.
`lessons_by_phase` is `{phase: [lesson ids in order]}`; return `{lesson id: split}`.

In [ ]:
PATTERN = ["dev", "dev", "dev", "test", "holdout"]  # 3 dev, 1 test, 1 holdout: 60/20/20

In [ ]:
def deal_by_phase(lessons_by_phase):
    """Each phase's lessons dealt across dev, test and holdout in the pattern above."""
    # TODO: for each phase, give its lessons PATTERN[0], PATTERN[1], ... in turn, starting again after 5
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_every_phase_with_five_lessons_reaches_every_split():
    dealt = deal_by_phase({"Frame": ["f1", "f2", "f3", "f4", "f5"], "Ship": ["s1", "s2", "s3", "s4", "s5", "s6"]})
    assert {dealt[l] for l in ["f1", "f2", "f3", "f4", "f5"]} == {"dev", "test", "holdout"}
    assert {dealt[l] for l in ["s1", "s2", "s3", "s4", "s5"]} == {"dev", "test", "holdout"}


def test_each_phase_starts_its_own_pattern():
    dealt = deal_by_phase({"A": ["a1"], "B": ["b1", "b2", "b3", "b4"]})
    assert dealt["a1"] == "dev" and dealt["b4"] == "test"


def test_the_pattern_repeats():
    dealt = deal_by_phase({"A": [f"a{i}" for i in range(1, 8)]})
    assert [dealt[f"a{i}"] for i in range(1, 8)] == ["dev", "dev", "dev", "test", "holdout", "dev", "dev"]

The real code (`tpe.assign`) does the same job a little differently: each new lesson goes to whichever
split its phase is **furthest short** of 60/20/20. The answer is then **frozen** in a file, so when lessons
are added, old ones never move: the goal of "Splits that don't move", reached by recording the split
instead of recomputing it.

In [ ]:
frozen = tpe.load_splits()
table = pd.crosstab(pd.Series([PHASE.get(l, "") for l in frozen], name="phase"), pd.Series(list(frozen.values()), name="split"))
table[["dev", "test", "holdout"]]

## 3. How sure is a pass rate?

A pass rate from 300 answers is not exact. The lesson "How many examples do you need?" gives a rule of
thumb: the margin is about 100 divided by the square root of the number of answers, in percentage points.
For 300 answers that's about plus or minus 6 points.

The **Wilson interval** (lesson "How sure is that pass rate?") does this properly. It stays between 0% and
100%, and it is sensible even for small groups or rates near 0% or 100%. In words: start from the observed
rate, pull it a little toward 50% (more for small samples), and spread out by an amount that shrinks as the
sample grows. `z = 1.96` is what makes it a 95% interval.

**Exercise.** Return `(rate, low, high)`. With `n = 0`, return `(0.0, 0.0, 0.0)`.

In [ ]:
def wilson(passed, n, z=1.96):
    """The pass rate and its Wilson 95% interval."""
    # TODO: p = passed / n; centre = (p + z*z/(2n)) / (1 + z*z/n);
    #       half = z * sqrt(p(1-p)/n + z*z/(4n*n)) / (1 + z*z/n); return p, centre - half, centre + half
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ninety_out_of_a_hundred():
    rate, low, high = wilson(90, 100)
    assert rate == 0.9 and round(low, 3) == 0.826 and round(high, 3) == 0.945


def test_nothing_measured():
    assert wilson(0, 0) == (0.0, 0.0, 0.0)


def test_it_never_leaves_zero_to_one():
    assert wilson(10, 10)[2] <= 1.0 and wilson(0, 10)[1] >= 0.0

## 4. Do two judges agree? Agreement versus kappa

Say two judges look at 100 answers. Judge A passes 96 and Judge B passes 98, and they agree on 96 of them:
**96% agreement**. It sounds great, but if almost everything passes, two judges that each said "pass" at
random would agree most of the time too.

**Cohen's kappa** (lesson "Do two judges agree?") answers: how much better than chance is the agreement?
1 is perfect, 0 is no better than chance. Above 0.6 is usually called "substantial".

- **Observed**: the share of answers where they agree.
- **Chance**: how often they'd agree by luck: (A's pass share x B's pass share) + (A's fail share x B's fail share).
- **Kappa** = (observed - chance) / (1 - chance). If chance is 1 (neither judge ever varies), kappa is undefined: return `None`.

**Exercise.** `a` and `b` are lists of True (pass) or False (fail), one per answer.

In [ ]:
def kappa(a, b):
    """Cohen's kappa between two judges' pass/fail verdicts."""
    # TODO: observed agreement, chance agreement, then (observed - chance) / (1 - chance); None if chance is 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ninety_six_percent_agreement_is_not_as_good_as_it_sounds():
    a = [True] * 96 + [False] * 4
    b = [True] * 98 + [False] * 2
    assert round(kappa(a, b), 2) == 0.66


def test_perfect_agreement():
    assert kappa([True, False, True, False], [True, False, True, False]) == 1.0


def test_no_variation_means_no_answer():
    assert kappa([True] * 5, [True] * 5) is None

## 5. A sample that keeps the rare cases

Judging all 1,025 dev answers takes about an hour at your workspace's rate limit, and dev is for trying
things, so a **sample** is the right tool. But Sonnet failed only 26 dev answers (about 2.5%). A plain random
300 would hold about 8 of them: too few to tell whether Haiku catches real problems.

So the sample is **stratified**: all 26 failures, then about 275 passed answers spread across phase and
question kind (`tpe.stratified_sample`). Because failures are over-represented on purpose, one overall number
would mislead. Measure each side on its own:

- **Catch rate**: of the answers the reference judge **failed**, the share the new judge **failed too**.
- **Keep rate**: of the answers the reference judge **passed**, the share the new judge **passed too**.

**Exercise.** Each answer is a dict with `passes` (the new judge) and `reference_pass`. Return
`{"catch": (caught, how many the reference failed), "keep": (kept, how many the reference passed)}`.

In [ ]:
def catch_and_keep(answers):
    """How often the new judge matches the reference on its failures, and on its passes."""
    # TODO: count reference failures, and those the new judge also failed; the same for passes
    raise NotImplementedError

In [ ]:
%%ipytest -qq

ROWS = [{"passes": False, "reference_pass": False}, {"passes": True, "reference_pass": False},
        {"passes": True, "reference_pass": True}, {"passes": False, "reference_pass": True}, {"passes": True, "reference_pass": True}]


def test_counts_each_side_on_its_own():
    assert catch_and_keep(ROWS) == {"catch": (1, 2), "keep": (2, 3)}


def test_nothing_to_catch():
    assert catch_and_keep([{"passes": True, "reference_pass": True}])["catch"] == (0, 0)

Now the real run. Haiku judged the 301-answer sample; Sonnet's verdicts are the reference.

In [ ]:
ck = catch_and_keep(REC["answers"])
for side, (hit, n) in ck.items():
    rate, low, high = wilson(hit, n)
    print(f"{side:5}: {hit} of {n} = {rate:.0%}   (95% interval {low:.0%} to {high:.0%})")

haiku = [a["passes"] for a in REC["answers"]]
sonnet = [a["reference_pass"] for a in REC["answers"]]
print(f"raw agreement {sum(x == y for x, y in zip(haiku, sonnet)) / len(haiku):.0%}, kappa {kappa(haiku, sonnet):.2f}")
print("the run's own numbers:", {k: REC["metrics"][k] for k in ("catch_rate", "keep_rate", "passes_kappa", "weighted_agreement")})
print("gate (kappa at least", REC["params"]["kappa_gate"], "):", "passed" if kappa(haiku, sonnet) >= float(REC["params"]["kappa_gate"]) else "failed")

Haiku catches most of what Sonnet failed (about 85%), but it **keeps only about a third** of what Sonnet
passed. Kappa is near 0: as a stand-in for Sonnet, Haiku fails the gate.

## 6. Where they disagree, and why

A single number hides where the problem is. Split it by phase and by kind of question:

In [ ]:
df = pd.DataFrame(REC["answers"])
df["phase"] = df.lesson_id.map(PHASE)
df["kind"] = df.key.map(tpe.kind_of)
df["agree"] = df.passes == df.reference_pass
for by in ("kind", "phase"):
    display(df.groupby(by).agg(answers=("agree", "size"), haiku_passes=("passes", "mean"), agreement=("agree", "mean")).round(2))

In [ ]:
haiku_only_fails = df[(~df.passes) & df.reference_pass]
print(f"Haiku failed {len(haiku_only_fails)} answers that Sonnet passed. Some of its reasons:")
for note in haiku_only_fails.notes.head(8):
    print(" -", note[:160])

Read those reasons. Some look **too strict**: terms the lesson itself teaches, flagged as jargon. But some
look **right**: words like MERGE, UDF or MCP used without explanation in a lesson that doesn't teach them, or
a small claim the lesson never makes. Sonnet let those through.

So the real question isn't "how do we make Haiku agree with Sonnet?" It's **which judge matches what you
want a beginner to read**. Only a person can settle that. That's the **golden set** from the lesson "Dev,
test and holdout sets": answers a person has checked by hand. Section 10 sets it up.

## 7. Is a difference real, or luck?

Next you'd change the judge's instructions (for example, "terms this lesson teaches count as explained") and
judge **the same 301 answers** again. To compare the two runs, only the answers where they **disagree** tell
you anything. That's McNemar's test (lesson "Real win or luck").

If the new judge gets `b` answers right that the old one got wrong, and the old one gets `c` right that the
new one got wrong, then with no real difference each disagreement is a coin flip. The p-value is the chance
of a split at least as lopsided as the one you saw. Under 0.05, call it real.

**Exercise.** The exact version: the chance of `min(b, c)` or fewer heads in `b + c` fair coin flips,
doubled (both directions count), and never more than 1. With no disagreements, return 1.0.
`math.comb(n, k)` counts the ways to get `k` heads in `n` flips; there are `2 ** n` outcomes in all.

In [ ]:
def mcnemar(b, c):
    """The exact two-sided McNemar p-value from the disagreements alone."""
    # TODO: n = b + c; add up math.comb(n, k) for k from 0 to min(b, c); divide by 2 ** n; double; cap at 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_twelve_against_eight_is_too_close_to_call():
    assert round(mcnemar(12, 8), 3) == 0.503


def test_fifteen_against_two_is_real():
    assert mcnemar(15, 2) < 0.01


def test_it_does_not_matter_which_side_is_which():
    assert mcnemar(3, 9) == mcnemar(9, 3)


def test_no_disagreements():
    assert mcnemar(0, 0) == 1.0

The project has the same function (`tpe.mcnemar`), and `uv run drill tutor prepared-compare RUN_A RUN_B`
runs it on two real judge runs, answer by answer.

## 8. What it cost

Every model call is counted. Cost is tokens times the price per million tokens (Anthropic's list prices;
Databricks bills in its own units, and the exact charge shows in `system.billing.usage` a few hours later).

In [ ]:
m = REC["metrics"]
price_in, price_out = tp.PRICES[REC["params"]["judge_model"]]
print(f"Haiku judging {int(m['judge_calls'])} answers: {int(m['judge_input_tokens']):,} tokens in, {int(m['judge_output_tokens']):,} out")
print(f"  = ${m['judge_cost_usd']:.2f}, about {100 * m['judge_cost_per_answer_usd']:.1f} cents an answer; "
      f"{int(m['judge_rate_limited'])} calls hit the rate limit and were retried")

# The drafting run, from the workspace's usage table (system.serving.endpoint_usage) on 2026-10-17
DRAFTING = {"databricks-claude-opus-5-5": (2_929_461, 1_187_505), "databricks-claude-sonnet-5-5": (3_429_008, 2_212_630)}
total = sum(tp.cost(model, tin, tout) for model, (tin, tout) in DRAFTING.items())
for model, (tin, tout) in DRAFTING.items():
    print(f"{model}: ${tp.cost(model, tin, tout):.2f}")
print(f"Drafting and judging 1,728 answers: ${total:.2f}, about {100 * total / 1728:.1f} cents an answer")

Notice how much is **output**: the judge writes a few lines of JSON, but these models reason before
answering, and reasoning is billed as output. Asking the judge to reason less is a lever worth measuring.

## 9. Where it all lives in MLflow

The evaluation follows MLflow's practices, so everything above can be found again and compared:

| What | Name | Why |
|---|---|---|
| **Datasets** | `tutor_prepared_dev`, `_test`, `_holdout`, `_dev_sample` | versioned: each run records the exact digest it scored |
| **Prompts** | `tutor_prepared_draft`, `tutor_prepared_judge` (alias `production`) | each run names the prompt versions it used |
| **Runs** | `tutor-prepared-<split>-<judge>-<commit>` | parameters, a pass rate per check with intervals, kappa, cost, and a gate tag |
| **Traces** | one per answer, tagged with split, phase and kind | the judge's verdicts are attached, with its reasons, tokens and time |
| **Tables** | `answers.json`, `by_stratum.json`, `strata.json` | per answer and per group, so a weak corner can't hide |

Set `LIVE = True` to read your own MLflow (the local server in `.env`, `http://localhost:5050`).

In [ ]:
LIVE = False

if LIVE:
    import mlflow

    mlflow.set_tracking_uri("http://localhost:5050")
    mlflow.set_experiment("daily-drill")
    runs = mlflow.search_runs(filter_string="tags.kind = 'tutor_prepared_eval' and attributes.status = 'FINISHED'")
    display(runs[["tags.mlflow.runName", "params.dataset", "params.dataset_digest", "params.judge_prompt", "metrics.keep_rate", "metrics.catch_rate", "tags.judge_gate"]])
    trace = mlflow.search_traces(filter_string="tags.kind = 'tutor_prepared_eval'", max_results=1, return_type="list")[0]
    for span in trace.data.spans:
        print(span.span_type, span.name, "| tags:", {k: v for k, v in trace.info.tags.items() if k in ("split", "phase", "question_kind")})
    for a in trace.info.assessments:
        print("  verdict:", a.name, a.value, "|", (a.rationale or "")[:80])
else:
    print("Replaying the recording. Set LIVE = True to read your local MLflow.")

## 10. Your turn: the golden set

Pick some answers where the two judges disagree, read each one, and say whether **you** would show it to a
beginner. Then see which judge agrees with you more. Twenty is enough to start; the lesson "How many
examples do you need?" tells you how sure twenty can make you (about plus or minus 22 points).

Fill in `MY_LABELS` as `{"lesson_id/key": True or False}`. True means "fine to show a beginner".

In [ ]:
disagreements = df[~df.agree].sample(n=min(20, int((~df.agree).sum())), random_state=7)
for _, row in disagreements.head(3).iterrows():  # print a few; read the rest the same way
    answer = ANSWERS[row.lesson_id][row.key]
    print(f"--- {row.lesson_id}/{row.key}   Haiku: {'pass' if row.passes else 'fail'}   Sonnet: {'pass' if row.reference_pass else 'fail'}")
    print(answer["q"])
    print(answer["a"][:600], "...")
    print("Haiku's reason:", row.notes[:200])
    print()

In [ ]:
MY_LABELS = {
    # "use_case_score/learn:term": True,
}

if MY_LABELS:
    mine = [(k, v) for k, v in MY_LABELS.items()]
    lookup = {f"{a['lesson_id']}/{a['key']}": a for a in REC["answers"]}
    you = [v for _, v in mine]
    print(f"Haiku agrees with you on {sum(lookup[k]['passes'] == v for k, v in mine)} of {len(mine)}")
    print(f"Sonnet agrees with you on {sum(lookup[k]['reference_pass'] == v for k, v in mine)} of {len(mine)}")
    print("When you're happy with your labels: `uv run drill tutor prepared --check` or `--reject lesson_id/key`")
else:
    print("Add some labels above to see which judge matches you.")

## What you used

| Idea | Lesson | Where in the project |
|---|---|---|
| Dev, test and holdout; stratify; golden set | Dev, test and holdout sets | `tpe.assign`, `prepared_splits.json` |
| Splits that never move | Splits that don't move | the frozen splits file |
| How many answers you need | How many examples do you need? | the 301-answer sample, the 374-answer test split |
| Pass rate with its margin | How sure is that pass rate? | `tpe.wilson`, every rate in the run |
| Agreement beyond chance | Do two judges agree? | `tpe.kappa`, the gate |
| Real win or luck | Real win or luck | `tpe.mcnemar`, `drill tutor prepared-compare` |
| A threshold to pass before you trust it | Must-pass gate | `KAPPA_GATE = 0.6`, the `judge_gate` tag |

The full write-up of how this was built is in `src/drill/tutor_prepared_eval.py` (its first comment) and
`docs/PAGE.md`.